# Exporting pipeline, step by step

Step 2 of 2: continues from `step1_run_sorting_stepbystep.ipynb`, once the sort is curated in Phy.

The second half of the pipeline, after curation in Phy. `run_exporting_pipeline.py` runs
these stages in this order, and this notebook calls the same verbs in the same order:

| stage | what it does | writes |
|---|---|---|
| `extract_sync` | the 1 Hz and 14 s burst pulses, from each system | edge files (`*.txt`, seconds) |
| `extract_lfp`, `extract_analog` | the LF band and the OneBox analog inputs, on their own clocks | `export/*.lfp.mat`, `*.analog.mat` |
| `time_remapping` | the map from probe time onto Blackrock time | `time_map.json`, spike times on Blackrock time |
| `validate_remapping` | the map checked on the held-out burst | `validation.json`, `sync_report.txt` |
| `export_results` | spike times, waveforms, quality metrics, figures | `export/` bundle |

After each verb runs, the notebook **reads back what it wrote** and shows how the result
was reached, using the same functions the pipeline calls. Nothing here is a second
implementation: if a number below disagrees with the file the pipeline wrote, that is a
bug worth reporting.

**Blackrock is the reference clock.** Neuropixels times are mapped onto Blackrock's
(the NSP's PTP clock, the axis of the `.nev` markers and eye traces), never the reverse.

Run from the `kilosort4` env: the waveform export reads the recording through Kilosort.
Section 4 also needs a session with Blackrock, since without a second clock there is
nothing to synchronize.

```bash
python run_exporting_pipeline.py --config <session>     # the same thing, in one go
```

In [ ]:
import json
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import spikesorting as ss
from spikesorting._export import metrics
from spikesorting._export.curated import load_phy_results
from spikesorting._sync import align, burst

# ---------------- edit these ----------------
SESSION = "session_template_1probe"      # the session config's name (configs/<name>.yaml)
MACHINE = {"darwin": "mac", "win32": "windows_rig"}.get(sys.platform, "hpc")

config = ss.load_session_config(SESSION, MACHINE)
_, npx = config.per_probe()[0]           # the first probe; a two-probe session repeats per probe
print(f"session {config.session} on {config.machine.name}")
print(f"  neuropixels: {npx.paths.sorted_for('neuropixels')}")
print(f"  blackrock:   {config.paths.sorted_for('blackrock') if config.has_data('blackrock') else 'none'}")
print(f"  synchronizes the two: {config.aligns_systems}")
print(f"  CatGT: {config.machine.catgt_dir or 'not here -> NumPy detector only'}")
print(f"  TPrime: {config.machine.tprime_dir or 'not here -> least-squares map'}")

## 1. Extract the sync pulses

Two pulse trains connect the clocks, and each is recorded on both systems:

| pulse | Neuropixels side | Blackrock side |
|---|---|---|
| **1 Hz square wave** | bit 6 of the probe's SY word (`npx_1hz`), and the same bit of the OneBox's SY word (`npx_obx_1hz`) | `.ns5` channel (`blackrock_1hz`) |
| **14 s coded burst** | OneBox analog input XA0 (`npx_burst`) | `.ns5` channel (`blackrock_burst`) |

The 1 Hz gives many edges, so it fits the map finely, but every edge looks the same, so
it cannot say *which* second is which. The burst comes every 14 s with its own pattern
of intra-burst intervals (its **code**), so it can.

Each edge file is one time per line, in seconds: from the stream's start for SpikeGLX,
on the NSP's PTP clock (~1.5e9 s) for Blackrock. CatGT and a NumPy detector both run
where CatGT is installed, and their outputs are compared.

In [ ]:
for cfg, system in ((npx, "neuropixels"), (config, "blackrock")):
    print(f"{system}: {ss.skip_reason(cfg, ss.extract_sync, system) or 'extracting...'}")
    if ss.skip_reason(cfg, ss.extract_sync, system):
        continue
    try:
        print("   ", ss.extract_sync(cfg, system).summary())
    except OSError as error:      # the share is not answering: read what is already on disk
        print(f"    recording not reachable ({error}); using the edge files already written")

In [ ]:
# Read the edge files back from disk, whoever wrote them.
edge_files = {
    "npx_1hz": npx.paths.sync_for("neuropixels") / "npx_1hz.txt",
    "npx_obx_1hz": npx.paths.sync_for("neuropixels") / "npx_obx_1hz.txt",
    "npx_burst": npx.paths.sync_for("neuropixels") / "npx_burst.txt",
    "blackrock_1hz": npx.paths.sync_for("blackrock") / "blackrock_1hz.txt",
    "blackrock_burst": npx.paths.sync_for("blackrock") / "blackrock_burst.txt",
}
edges = {name: np.loadtxt(path, ndmin=1) if path.exists() else np.empty(0)
         for name, path in edge_files.items()}

display(pd.DataFrame([
    {"edges": name, "n": t.size,
     "first (s)": t[0] if t.size else np.nan, "last (s)": t[-1] if t.size else np.nan,
     "span (min)": (t[-1] - t[0]) / 60 if t.size else np.nan, "file": str(edge_files[name])}
    for name, t in edges.items()
]).set_index("edges"))

**The 1 Hz intervals** should sit at 1.000 s with jitter of about one sample (33 µs at
30 kHz). A missed edge shows as a 2 s interval and an extra one as a short interval;
either one shifts the pairing by a whole cycle if it is not handled, which is why the
trains are paired *through a map* (section 4c), not by index.

In [ ]:
names_1hz = [n for n in ("npx_1hz", "npx_obx_1hz", "blackrock_1hz") if edges[n].size > 2]
fig, axes = plt.subplots(len(names_1hz), 1, figsize=(12, 2.2 * len(names_1hz)), squeeze=False)
for ax, name in zip(axes[:, 0], names_1hz):
    t = edges[name]
    intervals_ms = np.diff(t) * 1e3
    ax.plot((t[:-1] - t[0]) / 60, intervals_ms, ".", ms=2)
    ax.set_ylabel("interval (ms)")
    ax.set_title(f"{name}: median {np.median(intervals_ms):.4f} ms, jitter (SD) "
                 f"{np.std(intervals_ms[np.abs(intervals_ms - 1000) < 5]) * 1e3:.1f} us, "
                 f"{np.sum(np.abs(intervals_ms - 1000) > 5)} intervals off by > 5 ms", fontsize=9)
axes[-1, 0].set_xlabel("minutes from the first edge")
fig.tight_layout()
plt.show()

**The burst codes.** Each burst is a few pulses; the gaps between them are its code. The
same burst has the same code on both systems, to within a sample, whatever the clocks
are doing, because a code spans ~200 ms and drift cannot move it measurably over that.
Below: the first bursts of each recording, and their codes in ms.

In [ ]:
min_gap = config.burst_interval_s / 2          # a gap longer than this starts a new burst
codes = {}
for name in ("blackrock_burst", "npx_burst"):
    if edges[name].size:
        onsets = burst.group_burst_onsets(edges[name], min_gap)
        patterns = burst.burst_patterns(edges[name], min_gap)
        codes[name] = pd.DataFrame({
            "onset (s)": onsets[:6],
            "code (ms between pulses)": [np.round(p * 1e3, 2).tolist() for p in patterns[:6]],
        })
        print(f"{name}: {onsets.size} bursts")
        display(codes[name])
if not codes:
    print("no burst edges -- see 'without a burst' in section 4")

## 2. LFP and the OneBox analog inputs

Both are pure extraction: written on their own clock (`timebase: "stream"` /
`"onebox"`), with the Blackrock fields left NaN. `time_remapping` stamps
`t0_blackrock`/`fs_blackrock` into them afterwards without re-reading the samples. Each
is off unless the session says `export_lfp: true` / `export_analog: true`.

In [ ]:
for verb in (ss.extract_lfp, ss.extract_analog):
    why = ss.skip_reason(npx, verb, "neuropixels")
    print(f"{verb.__name__}: {why or verb(npx, 'neuropixels')}")

## 3. Synchronize: run it

`time_remapping` does all of section 4 in one call and writes `time_map.json`. Its log
lines say what it found at each step; section 4 then takes them one at a time.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("spikesorting").setLevel(logging.INFO)     # show the pipeline's own lines

time_map = ss.time_remapping(npx, "neuropixels")
print(time_map.summary() if time_map else ss.skip_reason(npx, ss.time_remapping, "neuropixels"))

map_path = npx.paths.aligned / "time_map.json"
saved = json.loads(map_path.read_text()) if map_path.exists() else {}
print(json.dumps({k: saved.get(k) for k in (
    "method", "cycles_from", "slope", "intercept", "drift_ppm", "n_points",
    "fit_max_residual_s", "burst_clock", "tprime_error")}, indent=2))

## 4. Synchronize, one step at a time

There are three clocks: the **probe** (headstage), the **OneBox**, and **Blackrock**.
The 1 Hz on the probe connects probe and Blackrock; the burst is recorded by the OneBox,
so it reaches Blackrock by OneBox → probe → Blackrock.

```
OneBox clock ──(4a: OneBox's own 1 Hz)──> probe clock ──(4b burst, then 4c 1 Hz)──> Blackrock clock
```

### 4a. OneBox → probe

Both carry the same 1 Hz in their SY words, sampled by two different crystals. A line
through the paired edges (`fit_same_run_map`) puts the burst on the probe clock. Without
this step the burst comparison would include the OneBox's own drift (`burst_clock:
"onebox"` in `time_map.json`).

In [ ]:
obx_map = None
if edges["npx_obx_1hz"].size and edges["npx_1hz"].size:
    obx_map = align.fit_same_run_map(edges["npx_obx_1hz"], edges["npx_1hz"], config.sync_period_s)
    print(f"OneBox -> probe: {obx_map.drift_ppm:+.3f} ppm, offset {obx_map.intercept * 1e3:.3f} ms, "
          f"{obx_map.n_points} edges, max residual {np.abs(obx_map.residuals_s).max() * 1e6:.1f} us")
    print(f"time_map.json says: {saved.get('onebox_to_probe')}")

    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.plot(np.arange(obx_map.residuals_s.size) / 60, obx_map.residuals_s * 1e6, ".", ms=2)
    ax.set_xlabel("minutes (edge number / 60)")
    ax.set_ylabel("residual (us)")
    ax.set_title("OneBox -> probe fit residuals: a flat band of about one sample means a clean fit")
    plt.show()
else:
    print("no OneBox 1 Hz (npx_obx_1hz.txt): the burst stays on the OneBox clock")

### 4b. The coarse map: bursts matched by code

Every burst on one system is compared with every burst on the other, and pairs whose
codes agree to 1 ms **vote** with their onset difference. The most popular offset wins
(a code that repeats hourly gives a second, much smaller cluster: the runner-up). The
winning pairs are then fitted with a **line**, so the coarse map already includes drift.

It used to be one global offset, scored at 5 ms. At 20 ppm that is wrong by 5 ms after
4 minutes, and once drift passed the scoring window a wrong 14 s cycle won by chance.
Pairing by code involves no offset at all, so drift cannot affect it.

In [ ]:
match = None
if edges["blackrock_burst"].size and edges["npx_burst"].size:
    npx_burst = obx_map.apply(edges["npx_burst"]) if obx_map is not None else edges["npx_burst"]
    match = burst.match_bursts(edges["blackrock_burst"], npx_burst, min_gap_s=min_gap)
    print(f"{match.n_matched} bursts matched (of {match.n_reference} Blackrock / {match.n_other} "
          f"probe); runner-up offset had {match.runner_up_matches} votes")
    print(f"coarse map: Blackrock = {match.slope:.9f} * probe + {match.intercept:.6f} "
          f"({(match.slope - 1) * 1e6:+.2f} ppm)")

    t_min = (match.other_matched - match.other_matched[0]) / 60
    fig, (ax_d, ax_r) = plt.subplots(1, 2, figsize=(13, 3.2))
    difference_ms = (match.reference_matched - match.other_matched - match.offset_s) * 1e3
    ax_d.plot(t_min, difference_ms, "o", ms=3, label="matched bursts")
    ax_d.plot(t_min, (match.apply(match.other_matched) - match.other_matched - match.offset_s) * 1e3,
              "-", label="fitted line")
    ax_d.set_xlabel("minutes")
    ax_d.set_ylabel(f"onset difference - {match.offset_s:.3f} s (ms)")
    ax_d.set_title("the clocks drift apart: the slope is the drift")
    ax_d.legend(fontsize=8)
    ax_r.plot(t_min, match.residuals_s * 1e3, "o", ms=3)
    ax_r.set_xlabel("minutes")
    ax_r.set_ylabel("residual (ms)")
    ax_r.set_title("burst onsets about the line")
    fig.tight_layout()
    plt.show()
else:
    print(f"no burst on one side; the cycle came from: {saved.get('cycles_from')}")
    print(json.dumps({"cycle_vote": saved.get("cycle_vote"),
                      "coarse_match_tried": saved.get("coarse_match_tried")}, indent=2))

**Why the burst has to come first.** The 1 Hz train alone fits equally well at every
whole-second shift, so the offset it gives can be wrong by whole cycles. The fit that
follows is self-consistent with whatever cycle it was handed, so a wrong cycle looks
perfect until validation. Here is the 1 Hz-only offset next to the burst's:

In [ ]:
if match is not None and edges["npx_1hz"].size and edges["blackrock_1hz"].size:
    naive = burst.estimate_offset(edges["blackrock_1hz"], edges["npx_1hz"], tolerance_s=0.1)
    t0 = edges["npx_1hz"][0]
    from_burst = match.apply(t0) - t0
    print(f"offset from the 1 Hz alone : {naive:.6f} s")
    print(f"offset from the bursts     : {from_burst:.6f} s  (at the first probe edge)")
    print(f"difference                 : {naive - from_burst:+.3f} s "
          f"= {round(naive - from_burst)} whole cycle(s)")

### 4c. The fine map: 1 Hz edges paired through the coarse line

Both 1 Hz trains are trimmed to the stretch both recorded (an edge only one side has
would shift the pairing by a cycle), each probe edge is put on Blackrock time by the
coarse line and paired with the Blackrock edge within a quarter period, and a line is
fitted through all the pairs. That line is `time_map.json`'s `slope`/`intercept`.

Pairing goes through the coarse *line*, not its offset: 50 ppm over 3 h is 540 ms, past
the ±250 ms pairing window.

In [ ]:
fit = None
if edges["npx_1hz"].size and edges["blackrock_1hz"].size and saved:
    coarse = (match.apply(edges["npx_1hz"]) if match is not None   # what time_remapping used
              else saved["slope"] * edges["npx_1hz"] + saved["intercept"])
    br_trim, npx_trim, npx_coarse = align.trim_pair_under_map(
        edges["blackrock_1hz"], edges["npx_1hz"], coarse, margin_s=config.sync_period_s / 4)
    ref_idx, other_idx = burst.match_times(br_trim, npx_coarse, 0.0, tolerance_s=config.sync_period_s / 4)
    fit = align.fit_linear_map(npx_trim[other_idx], br_trim[ref_idx])

    print(f"trimmed: Blackrock {edges['blackrock_1hz'].size} -> {br_trim.size}, "
          f"probe {edges['npx_1hz'].size} -> {npx_trim.size}; {ref_idx.size} pairs")
    print(f"recomputed : slope {fit.slope:.12f}, intercept {fit.intercept:.6f}, {fit.drift_ppm:+.3f} ppm")
    print(f"saved      : slope {saved['slope']:.12f}, intercept {saved['intercept']:.6f}, "
          f"{saved['drift_ppm']:+.3f} ppm")

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot((npx_trim[other_idx] - npx_trim[0]) / 60, fit.residuals_s * 1e6, ".", ms=2)
    ax.set_xlabel("minutes")
    ax.set_ylabel("residual (us)")
    ax.set_title(f"1 Hz pairs about the line: max {np.abs(fit.residuals_s).max() * 1e6:.0f} us. "
                 "Slow structure here is the crystals wandering, which a line cannot follow")
    plt.show()

### 4d. TPrime: following the wander

The residuals above are not noise: the two crystals wander relative to each other on a
scale of minutes. TPrime follows that by giving each 1 Hz interval its own offset
(`out = t + (to_k - from_k)`, with `k` the last edge before `t`). It is used for spike
times where TPrime is installed; otherwise the line above is applied
(`method: linear_fit`). The analog and LFP exports always use the line.

Below: how far TPrime moved each spike from where the line would have put it.

In [ ]:
results_dir = npx.paths.sorted_for("neuropixels")
raw_path = results_dir / "neuropixels_spike_seconds.npy"
mapped_path = results_dir / "neuropixels_spike_seconds_blackrock.npy"
print(f"method: {saved.get('method')}" + (f" (TPrime failed: {saved['tprime_error']})"
                                         if saved.get("tprime_error") else ""))
if saved.get("method") == "tprime" and raw_path.exists() and mapped_path.exists():
    raw = np.load(raw_path, mmap_mode="r")
    mapped = np.load(mapped_path, mmap_mode="r")
    pick = np.linspace(0, raw.size - 1, min(raw.size, 200_000)).astype(int)
    tprime_minus_line_ms = (mapped[pick] - (saved["slope"] * raw[pick] + saved["intercept"])) * 1e3
    print(f"TPrime - line: median {np.median(tprime_minus_line_ms):+.3f} ms, "
          f"range {tprime_minus_line_ms.min():+.3f} to {tprime_minus_line_ms.max():+.3f} ms")
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(raw[pick] / 60, tprime_minus_line_ms, ",")
    ax.set_xlabel("minutes into the probe recording")
    ax.set_ylabel("TPrime - line (ms)")
    plt.show()

## 5. Validate on the held-out bursts

The map was fitted on the 1 Hz only, so the burst onsets are **held out**: mapped
OneBox → probe → Blackrock, each should land on Blackrock's own onset to within
`alignment_tolerance_s` (1 ms). That is a real test, which the 1 Hz residuals above are
not. Uncorrected drift of 20 ppm is 72 ms after an hour, so a wrong slope fails here.

`sync_report.txt` puts this together with every other signal recorded on both systems.
For the traces themselves, see `compare_sync_analog.ipynb`.

In [ ]:
validation = ss.validate_remapping(npx, npx.export_figures)
if validation is not None and hasattr(validation, "residuals_s") and validation.n_checked:
    print(validation.summary())
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot((validation.times_s - validation.times_s[0]) / 60, validation.residuals_s * 1e3, "o", ms=3)
    for sign in (1, -1):
        ax.axhline(sign * validation.tolerance_s * 1e3, color="r", ls="--", lw=0.8)
    ax.set_xlabel("minutes")
    ax.set_ylabel("held-out residual (ms)")
    ax.set_title("each burst onset, mapped, minus Blackrock's (red: tolerance)")
    plt.show()
else:
    print(validation.summary() if validation is not None else "nothing to validate")

report_txt = npx.paths.export_for("neuropixels") / "sync_report.txt"
print(report_txt.read_text() if report_txt.exists() else "no sync_report.txt")

## 6. Export

`export_results` reads the sorting (Phy's labels where curated; every cluster not
labelled noise is exported), puts spike times on Blackrock time, measures each unit's
mean waveform from the recording, computes the metrics and writes the bundle.

In [ ]:
exports = {}
for cfg, system in ((npx, "neuropixels"), (config, "blackrock")):
    why = ss.skip_reason(cfg, ss.export_results, system)
    exports[system] = None if why else ss.export_results(cfg, system)
    result = exports[system]
    if why:
        print(f"{system}: {why}")
    elif result:
        print(f"{system}: {result['n_units']} units, timebase {result['timebase']}")
    else:
        print(f"{system}: no sorting to export")

export_dir = npx.paths.export_for("neuropixels")
units = pd.read_csv(export_dir / "units.csv") if (export_dir / "units.csv").exists() else None
if units is not None:
    display(units.head(10))

## 7. Quality metrics, recomputed by hand

Every `units.csv` column for one unit, recomputed below from the exported spike times
(`sorted_spikes.mat`, on Blackrock time) with plain NumPy, and compared with what the
export wrote:

| column | how |
|---|---|
| `n_spikes`, `firing_rate_hz` | count; count / recording window. The window is first to last spike over **all** exported units, on the timebase in use |
| `presence_ratio` | the window in 100 equal bins: fraction of bins with at least one spike. A unit that drifts away mid-session drops below 1 |
| `isi_median_ms`, `isi_fraction` | inter-spike intervals; fraction shorter than 1.5 ms (refractory violations). A single unit should be well under 1 % |
| `amp_cv`, `amp_slope_per_hour`, `amp_median` | Kilosort's per-spike amplitude (`amplitudes.npy`, template scaling, arbitrary units): SD / mean, and a line through amplitude vs time, per hour, normalised by the median |
| `trough_amplitude`, `peak_amplitude`, `peak_to_trough_ms`, `half_width_ms` | from the **measured** mean waveform on the peak channel, in µV: the minimum, the maximum after it, the time between, and the trough's width at half depth |

Change `UNIT` to look at another unit (`None` picks the good unit with most spikes).

In [ ]:
UNIT = None

mat_path = next(export_dir.glob("*.sorted_spikes.mat"), None)
assert units is not None and mat_path is not None, "nothing exported yet -- run section 6 first"
with h5py.File(mat_path, "r") as handle:
    s = handle["sorted_spikes"]
    all_times = s["TimeStamps"][()].ravel()
    all_units = s["Unit"][()].ravel().astype(int)
    mean_waves = s["MeanWaveform"][()] if "MeanWaveform" in s else None          # (n_samples, n_units)
    std_waves = s["StdWaveform"][()] if "StdWaveform" in s else None
    wave_units = s["info"]["MeanWaveform_Unit_No"][()].ravel().astype(int) if mean_waves is not None else []
    fs = float(s["TimeRes"][()].ravel()[0])

good = units[units["label"] == "good"] if (units["label"] == "good").any() else units
unit_id = int(good.sort_values("n_spikes").iloc[-1]["unit_id"]) if UNIT is None else UNIT
row = units.set_index("unit_id").loc[unit_id]
times = np.sort(all_times[all_units == unit_id])

window_start, window_stop = all_times.min(), all_times.max()
duration = window_stop - window_start
isi = np.diff(times)
presence_bins = np.histogram(times, bins=np.linspace(window_start, window_stop, 101))[0]
phy = load_phy_results(npx.paths.sorted_for("neuropixels"))
amps = phy.amplitudes_for(unit_id)
amp_slope = np.polyfit(times, amps, 1)[0] if amps is not None and amps.size == times.size else np.nan

by_hand = {
    "n_spikes": times.size,
    "firing_rate_hz": times.size / duration,
    "presence_ratio": np.count_nonzero(presence_bins) / 100,
    "isi_median_ms": np.median(isi) * 1e3,
    "isi_fraction": np.mean(isi < 1.5e-3),
    "amp_cv": np.std(amps) / np.mean(amps) if amps is not None else np.nan,
    "amp_slope_per_hour": amp_slope * 3600 / np.median(amps) if amps is not None else np.nan,
    "amp_median": np.median(amps) if amps is not None else np.nan,
}

wave = std = None
if mean_waves is not None and unit_id in list(wave_units):
    column = list(wave_units).index(unit_id)
    wave, std = mean_waves[:, column], std_waves[:, column]
    trough = int(np.argmin(wave))
    peak = trough + int(np.argmax(wave[trough:]))          # the largest point *after* the trough
    # Half-width: walk out from the trough to the last samples at or below half its
    # depth, then interpolate each crossing between that sample and the next one out.
    half = wave[trough] / 2
    left = right = trough
    while left > 0 and wave[left - 1] <= half:
        left -= 1
    while right < wave.size - 1 and wave[right + 1] <= half:
        right += 1
    left_x = left - (half - wave[left]) / (wave[left - 1] - wave[left])
    right_x = right + (half - wave[right]) / (wave[right + 1] - wave[right])
    by_hand.update({
        "trough_amplitude": wave[trough],
        "peak_amplitude": wave[peak],
        "peak_to_trough_ms": (peak - trough) / fs * 1e3,
        "half_width_ms": (right_x - left_x) / fs * 1e3,
    })

comparison = pd.DataFrame({"by hand": pd.Series(by_hand), "units.csv": row.reindex(list(by_hand))})
comparison["agree"] = np.isclose(comparison["by hand"].astype(float),
                                 comparison["units.csv"].astype(float), rtol=1e-3, equal_nan=True)
print(f"unit {unit_id} ({row['label']}), channel {row['channel']}, timebase of the times: "
      f"{'Blackrock' if window_start > 1e8 else 'the sorter'}")
display(comparison)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

ax = axes[0, 0]
ax.hist(isi * 1e3, bins=np.arange(0, 50.5, 0.5), color="tab:blue")
ax.axvline(1.5, color="r", ls="--", lw=1)
ax.set_xlabel("inter-spike interval (ms)")
ax.set_ylabel("count")
ax.set_title(f"ISI: {by_hand['isi_fraction'] * 100:.2f}% under 1.5 ms (red)")

ax = axes[0, 1]
centres = np.linspace(window_start, window_stop, 101)
ax.bar((centres[:-1] - window_start) / 60, presence_bins / (duration / 100),
       width=duration / 100 / 60, color="tab:blue")
ax.set_xlabel("minutes into the recording window")
ax.set_ylabel("rate (Hz)")
ax.set_title(f"firing rate in the 100 presence bins: {by_hand['presence_ratio']:.2f} non-empty")

ax = axes[1, 0]
if amps is not None:
    pick = np.linspace(0, times.size - 1, min(times.size, 20_000)).astype(int)
    ax.plot((times[pick] - window_start) / 60, amps[pick], ",", color="tab:blue")
    ax.plot((times - window_start) / 60, np.polyval(np.polyfit(times, amps, 1), times), "r-")
    ax.set_xlabel("minutes")
    ax.set_ylabel("Kilosort amplitude (a.u.)")
    ax.set_title(f"amplitude: CV {by_hand['amp_cv']:.2f}, "
                 f"{by_hand['amp_slope_per_hour'] * 100:+.1f}% of median per hour")

ax = axes[1, 1]
if wave is not None:
    t_ms = (np.arange(wave.size) - wave.size // 2) / fs * 1e3
    ax.fill_between(t_ms, wave - std, wave + std, color="tab:blue", alpha=0.2)
    ax.plot(t_ms, wave, color="tab:blue")
    ax.plot(t_ms[trough], wave[trough], "v", color="r")
    ax.plot(t_ms[peak], wave[peak], "^", color="r")
    ax.axhline(wave[trough] / 2, color="grey", ls=":", lw=1)
    ax.set_xlabel("ms from the spike")
    ax.set_ylabel(row.get("waveform_units", "") or "amplitude")
    ax.set_title(f"measured mean +/- SD: peak-to-trough {by_hand['peak_to_trough_ms']:.2f} ms")
else:
    ax.text(0.5, 0.5, "no measured waveform\n(binary not reachable at export)",
            ha="center", va="center", transform=ax.transAxes)

fig.suptitle(f"unit {unit_id}")
fig.tight_layout()
plt.show()

## 8. Where the time went

`run_exporting_pipeline.py` adds the exporting phase to `run_info.json` and writes
`export/timing.txt`; here, only the sorting run's phases are on record, since the
notebook's stages were not timed by the runner.

In [ ]:
print(ss.timing_table(npx, "neuropixels") or "no run_info.json timing yet -- sort first")